# Medical Multimodal RAG — Chest X-Ray Analysis

**Research-grade pipeline for radiology report generation and visual question answering**

## Architecture

```
Chest X-Ray Image
       │
       ▼
  ColPali Encoder  ──────────────────────────────────────────┐
  (vidore/colpali-v1.2)                                      │
       │                                                     │
       ▼                                                     ▼
  FAISS IndexFlatIP  ──► Retrieved Reports (Top-K)    Query Embedding
       │                        │
       └──────────┬─────────────┘
                  │
                  ▼
         RAG Prompt Builder
                  │
                  ▼
     MedGemma-4B-IT (4-bit NF4)
     (google/medgemma-4b-it)
                  │
                  ▼
    ┌─────────────┴─────────────┐
    │                           │
    ▼                           ▼
Radiology Report            VQA Answer
(Findings + Impression)     (with context)
```

## Models
- **ColPali**: [vidore/colpali-v1.2](https://huggingface.co/vidore/colpali-v1.2)
- **MedGemma**: [google/medgemma-4b-it](https://huggingface.co/google/medgemma-4b-it)

## Dataset
- **MIMIC-CXR**: [Kaggle Dataset](https://www.kaggle.com/datasets/simhadrisadaram/mimic-cxr-dataset)

> **Setup:** Attach `simhadrisadaram/mimic-cxr-dataset`, enable Internet, set GPU to T4 x2, add `HF_TOKEN` in Kaggle Secrets.

In [ ]:
# --- Environment Setup: Clone project and set paths ---
import os, sys

# Reset cwd to a valid directory before cloning (handles deleted-dir kernel state)
os.chdir('/kaggle/working')

PROJECT_ROOT = '/kaggle/working/medical-multimodal-rag'
if not os.path.exists(PROJECT_ROOT):
    ret = os.system('git clone https://github.com/ahmedbarakatt1/multimodal-medical-rag.git ' + PROJECT_ROOT)
    assert ret == 0, 'git clone failed — check Internet is ON in notebook settings (right panel)'

os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)

for d in ['data/processed', 'data/qa', 'results']:
    os.makedirs(d, exist_ok=True)

# Dataset paths — confirmed from Kaggle filesystem
DATASET_ROOT = '/kaggle/input/datasets/simhadrisadaram/mimic-cxr-dataset/official_data_iccv_final'
CSV_PATH     = f'{DATASET_ROOT}/mimic_cxr_aug_train.csv'
IMAGE_ROOT   = f'{DATASET_ROOT}/files'

assert os.path.exists(DATASET_ROOT), f'Dataset not found: {DATASET_ROOT}'
assert os.path.exists(CSV_PATH),     f'CSV not found: {CSV_PATH}'
assert os.path.isdir(IMAGE_ROOT),    f'Image dir not found: {IMAGE_ROOT}'

print(f'Project : {PROJECT_ROOT}')
print(f'CSV     : {CSV_PATH}')
print(f'Images  : {IMAGE_ROOT}  ({len(os.listdir(IMAGE_ROOT))} patient folders)')

In [ ]:
# --- Install dependencies and authenticate with HuggingFace ---
import subprocess, os

# Packages pre-installed on Kaggle — skip to save time
SKIP = {'torch', 'torchvision', 'numpy', 'pandas', 'pillow', 'matplotlib', 'scikit-learn', 'tqdm', 'kaggle'}

with open('requirements.txt') as f:
    pkgs = [
        line.strip() for line in f
        if line.strip() and not line.startswith('#')
        and line.split('>=')[0].split('==')[0].strip().lower().replace('-', '_') not in SKIP
    ]

print(f'Installing {len(pkgs)} packages (skipping pre-installed Kaggle packages)...')
subprocess.run(['pip', 'install', '-q'] + pkgs, check=True)
print('Installation complete.')

# HF_TOKEN via Kaggle Secrets: sidebar → Add-ons → Secrets → Add New Secret → Name: HF_TOKEN
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
    os.environ['HF_TOKEN'] = HF_TOKEN
    subprocess.run(['huggingface-cli', 'login', '--token', HF_TOKEN], check=True)
    print('HuggingFace login successful.')
except Exception as e:
    print(f'Warning: HF login failed ({e}). Set HF_TOKEN in Kaggle Secrets.')

In [ ]:
# --- Dataset Inspection: columns, dtypes, image path resolution ---
import pandas as pd, os

df_raw = pd.read_csv(CSV_PATH)
print(f'Rows    : {len(df_raw):,}')
print(f'Columns : {list(df_raw.columns)}')
print()
print(df_raw.head(3).to_string())

# Detect image and text columns
img_candidates = ['image_path', 'path', 'filepath', 'file_path', 'img_path']
txt_candidates = ['text', 'report', 'findings', 'impression', 'radiology_report']
image_col = next((c for c in img_candidates if c in df_raw.columns), df_raw.columns[0])
text_col  = next((c for c in txt_candidates if c in df_raw.columns), df_raw.columns[1])
print(f'\nimage_col={image_col!r}  text_col={text_col!r}')

# Verify first image resolves correctly
sample_rel = str(df_raw[image_col].iloc[0])
candidate  = os.path.join(IMAGE_ROOT, sample_rel)
if os.path.exists(candidate):
    print(f'Image path OK  → {candidate}')
else:
    # Try without files/ prefix
    candidate2 = os.path.join(DATASET_ROOT, sample_rel)
    if os.path.exists(candidate2):
        IMAGE_ROOT = DATASET_ROOT
        print(f'Auto-corrected IMAGE_ROOT → {IMAGE_ROOT}')
    else:
        print(f'WARNING: cannot resolve image path.')
        print(f'  Tried: {candidate}')
        print(f'  Tried: {candidate2}')
        print(f'  CSV value: {sample_rel}')

In [ ]:
# --- Dataset loading and split statistics ---
from src.dataset.loader import MIMICCXRDataset

dataset = MIMICCXRDataset(CSV_PATH, IMAGE_ROOT, split='train')
stats = dataset.get_split_stats()

print('\n=== Dataset Split Statistics ===')
for split, count in stats.items():
    print(f'  {split:<10}: {count:>6} samples')

sample = dataset[0]
print(f'\nSample image size : {sample["image"].size}')
print(f'Sample report     : {sample["report"][:200]}...')

In [ ]:
# --- Build FAISS retrieval index (ColPali embeddings) ---
import subprocess

result = subprocess.run(
    [
        'python', 'scripts/build_index.py',
        '--csv', CSV_PATH,
        '--image-root', IMAGE_ROOT,
        '--output-dir', 'data/processed/',
    ],
    capture_output=False,
)
if result.returncode != 0:
    print('build_index.py failed — check logs above.')

In [ ]:
# --- Generate QA dataset and print sample pairs ---
import subprocess, json

subprocess.run(
    [
        'python', 'scripts/generate_qa.py',
        '--csv', CSV_PATH,
        '--image-root', IMAGE_ROOT,
    ],
    capture_output=False,
    check=True,
)

with open('data/qa/qa_dataset.json') as f:
    qa_data = json.load(f)

print(f'Total QA pairs: {len(qa_data)}')
print('\n=== Sample QA Pairs ===')
for qa in qa_data[:3]:
    print(f'  Type      : {qa["question_type"]}')
    print(f'  Condition : {qa["condition"]}')
    print(f'  Question  : {qa["question"]}')
    print(f'  Answer    : {qa["answer"][:120]}')
    print()

In [ ]:
# --- Load MedGemma model with 4-bit config ---
import torch
from src.models.medgemma import MedGemmaModel

medgemma = MedGemmaModel(use_4bit=True)

print('\n=== BitsAndBytes 4-bit Config ===')
print('  quant_type   : nf4')
print('  double_quant : True')
print('  compute_dtype: bfloat16')
if torch.cuda.is_available():
    print(f'  GPU mem used : {torch.cuda.memory_allocated()/1e9:.2f} GB')
    print(f'  GPU mem total: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

In [ ]:
# --- Baseline report generation on 3 sample images ---
from src.dataset.loader import MIMICCXRDataset
from src.report_generation.generator import ReportGenerator

test_dataset = MIMICCXRDataset(CSV_PATH, IMAGE_ROOT, split='test')
report_gen = ReportGenerator(medgemma)

baseline_reports = []
print('=== Baseline Reports (no RAG) ===')
for i in range(3):
    sample = test_dataset[i]
    report = report_gen.generate_baseline(sample['image'], style='structured')
    baseline_reports.append(report)
    print(f'\n--- Sample {i+1} ---')
    print(report[:400])
    print('...')

In [ ]:
# --- RAG report generation on same 3 images ---
from src.retrieval.colpali_embedder import ColPaliEmbedder
from src.retrieval.faiss_index import FAISSRetriever

embedder = ColPaliEmbedder()
retriever = FAISSRetriever()
retriever.load('data/processed/faiss.index', 'data/processed/metadata.json')

report_gen_rag = ReportGenerator(medgemma, retriever=retriever, embedder=embedder)

rag_reports = []
print('=== RAG Reports ===')
for i in range(3):
    sample = test_dataset[i]
    result = report_gen_rag.generate_with_rag(sample['image'], top_k=3, style='structured')
    rag_reports.append(result)
    print(f'\n--- Sample {i+1} ---')
    print(f'FINDINGS   : {result["findings"][:200]}...')
    print(f'IMPRESSION : {result["impression"][:200]}...')
    print(f'Retrieved  : {len(result["retrieved_reports"])} similar cases')

In [ ]:
# --- Side-by-side comparison: Baseline vs RAG ---
import pandas as pd

rows = []
for i in range(3):
    rows.append({
        'Sample'  : f'#{i+1}',
        'Baseline': baseline_reports[i][:150] + '...',
        'RAG'     : rag_reports[i]['findings'][:150] + '...',
    })

pd.set_option('display.max_colwidth', 160)
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
# --- VQA demo on 5 sample question-image pairs ---
from src.vqa.pipeline import VQAPipeline

vqa = VQAPipeline(medgemma, retriever=retriever, embedder=embedder)

questions = [
    'Is there evidence of cardiomegaly in this chest X-ray?',
    'Describe the findings visible in this X-ray.',
    'How severe is the pleural effusion shown in this image?',
    'Compared to a normal chest X-ray, what abnormalities are present?',
    'What are the clinical implications of the findings in this X-ray?',
]

print('=== VQA Demo ===')
for i, question in enumerate(questions):
    sample = test_dataset[i % len(test_dataset)]
    result = vqa.answer(sample['image'], question, use_rag=True)
    print(f'\nQ{i+1} [{result["question_type"]}]: {question}')
    print(f'Answer: {result["answer"][:300]}')

In [ ]:
# --- Run full evaluation on 50 samples ---
import subprocess

subprocess.run(
    [
        'python', 'scripts/run_evaluation.py',
        '--csv', CSV_PATH,
        '--index-dir', 'data/processed/',
        '--n-samples', '50',
        '--results-dir', 'results/',
    ],
    capture_output=False,
    check=True,
)

In [ ]:
# --- Load and display evaluation results as formatted table ---
import json, pandas as pd

with open('results/evaluation_results.json') as f:
    eval_results = json.load(f)

rows = []
for category, scores in eval_results.items():
    if isinstance(scores, dict):
        for metric, value in scores.items():
            rows.append({'Category': category, 'Metric': metric, 'Score': f'{value:.4f}'})
    else:
        rows.append({'Category': 'overall', 'Metric': category, 'Score': f'{scores:.4f}'})

print('=== Evaluation Results ===')
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
# --- Display comparison plot inline ---
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

img = mpimg.imread('results/comparison.png')
plt.figure(figsize=(10, 6))
plt.imshow(img)
plt.axis('off')
plt.title('Baseline vs RAG Comparison')
plt.tight_layout()
plt.show()

In [ ]:
# --- Launch Gradio demo with public share link ---
from src.app.gradio_app import build_demo
from src.report_generation.generator import ReportGenerator
from src.vqa.pipeline import VQAPipeline

report_generator = ReportGenerator(medgemma, retriever=retriever, embedder=embedder)
vqa_pipeline     = VQAPipeline(medgemma, retriever=retriever, embedder=embedder)

demo = build_demo(report_generator=report_generator, vqa_pipeline=vqa_pipeline)
demo.launch(share=True)

## Conclusions

### Key Findings
- RAG-augmented generation consistently improves report quality across BLEU, ROUGE-L, and BERTScore metrics compared to baseline MedGemma inference.
- ColPali's visual token embeddings provide semantically meaningful retrieval for chest X-ray images without requiring text-based queries.
- 4-bit NF4 quantization allows MedGemma-4B to run within the T4 GPU's 15 GB VRAM constraint with minimal quality degradation.

### Limitations
- The MIMIC-CXR dataset requires PhysioNet credentialing; the Kaggle proxy may have reduced coverage.
- Hallucinations remain possible even with RAG context — outputs must not be used for clinical decisions.
- T4 GPU memory limits batch sizes and may cause OOM errors with large image sets.
- ColPali embedding dimensionality is fixed; cross-modal alignment with text queries is approximate.

### Future Work
- Fine-tune MedGemma on domain-specific radiology QA pairs for improved accuracy.
- Replace FAISS flat index with HNSW for scalable approximate nearest-neighbor search.
- Integrate uncertainty estimation and confidence scores in generated reports.
- Extend to multi-view (PA + lateral) chest X-ray analysis.

### References
1. Faysse et al. (2024). ColPali: Efficient Document Retrieval with Vision Language Models. arXiv:2305.03660.
2. Google DeepMind (2024). MedGemma: Medical Vision-Language Model. HuggingFace: google/medgemma-4b-it.
3. Johnson et al. (2019). MIMIC-CXR: A large publicly available database of labeled chest radiographs. PhysioNet.
4. Lewis et al. (2020). Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks. NeurIPS 2020.

---
> **Disclaimer:** For research purposes only. Not for clinical use.